<div align="center" style="background:linear-gradient(135deg,#0A0A0A 0%,#1A1A1A 50%,#0D0D0D 100%);border:1.5px solid #C9A84C;border-radius:18px;padding:34px 24px 30px;margin:10px 0 20px;box-shadow:0 0 36px rgba(201,168,76,0.18),0 8px 32px rgba(0,0,0,0.6);font-family:Arial,sans-serif;text-align:center;"><div style="width:65%;height:3px;margin:0 auto 22px;background:linear-gradient(90deg,transparent,#C9A84C,#FFD700,#C9A84C,transparent);border-radius:4px;"></div><p align="center" style="color:#C9A84C;letter-spacing:5px;font-size:11px;font-weight:700;text-transform:uppercase;margin:0 0 10px;text-align:center;">&#10022; TUSHKOSH 2.0 PRESENTS &#10022;</p><h1 align="center" style="color:#FFFFFF;font-size:2.05em;margin:0 0 8px;text-shadow:0 2px 12px rgba(201,168,76,0.4);letter-spacing:1px;text-align:center;">Qwen 3 Tts 1 7B</h1><div style="width:80px;height:2px;margin:12px auto;background:linear-gradient(90deg,transparent,#C9A84C,#FFD700,#C9A84C,transparent);border-radius:2px;"></div><p align="center" style="color:#C9A84C;font-size:13px;font-weight:700;letter-spacing:3px;margin:0 0 22px;text-transform:uppercase;text-align:center;">Learn &bull; Build &bull; Create</p><div align="center" style="text-align:center;margin:0 0 18px;"><a href="https://www.youtube.com/@TushKosh2.0?sub_confirmation=1" target="_blank" style="display:inline-block;background:linear-gradient(135deg,#C9A84C 0%,#FFD700 50%,#C9A84C 100%);color:#000000;font-weight:800;font-size:13px;padding:12px 30px;border-radius:50px;text-decoration:none;letter-spacing:1px;box-shadow:0 4px 20px rgba(201,168,76,0.5);">&#9654; Subscribe to TushKosh 2.0</a></div><div align="center" style="text-align:center;margin-top:6px;"><img src="https://img.shields.io/badge/TushKosh%202.0-AI%20Tutorials-C9A84C?style=flat-square&labelColor=0A0A0A" style="margin:3px;vertical-align:middle;" /><img src="https://img.shields.io/badge/Original%20by-AIQUEST%20Academy-555555?style=flat-square&labelColor=1A1A1A" style="margin:3px;vertical-align:middle;" /></div><div style="width:65%;height:2px;margin:22px auto 0;background:linear-gradient(90deg,transparent,#C9A84C,#FFD700,#C9A84C,transparent);border-radius:4px;"></div></div>

In [ ]:
#@title 📦 Step 1: Environment Setup & Fast Dependency Install
import os
import sys
import gc
import psutil

print("=" * 60)
print("🎙️ Qwen3-TTS 1.7B - Colab T4 Environment Setup")
print("📺 Created by: TushKosh 2.0")
print("🔗 YouTube: @TushKosh2.0 | X: @TushKosh2.0")
print("=" * 60)

# 1. Memory, Hub and OS cache optimization
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.6"
os.environ["MALLOC_TRIM_THRESHOLD_"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

try:
    os.system("echo 3 | sudo tee /proc/sys/vm/drop_caches > /dev/null 2>&1")
    os.system("echo 1 | sudo tee /proc/sys/vm/overcommit_memory > /dev/null 2>&1")
except Exception:
    pass
gc.collect()

# 2. Hardware and GPU check
import torch
print(f"🔧 Python: {sys.version.split()[0]}")
print(f"🔧 PyTorch: {torch.__version__}")
print(f"🔧 RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB total | {psutil.virtual_memory().available / 1024**3:.1f} GB free")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✅ GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
    print(f"✅ CUDA Version: {torch.version.cuda}")
else:
    print("\n⚠️ WARNING: No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

print("\n📦 Installing optimized dependencies (qwen-tts, transformers 4.57.3, pyloudnorm, hf-transfer, sox)...")
# Install system sox binary to eliminate 'SoX could not be found' warning
!apt-get -y -qq install sox libsox-fmt-all > /dev/null 2>&1
# Single streamlined command: qwen-tts strictly requires transformers==4.57.3
!pip install -q "transformers==4.57.3" "accelerate>=1.12.0" soundfile "gradio>=5.0.0" pyloudnorm hf-transfer qwen-tts

print("\n✅ Environment setup completed successfully!")

In [ ]:
#@title 🚀 Step 2: Load Model & Launch Gradio Web UI
import os
import gc
import time
import tempfile
import warnings
import logging
import io
import contextlib
import numpy as np
import soundfile as sf
import torch
import gradio as gr
import pyloudnorm as pyln

# ── Silence Framework, Starlette, Hub, & Loudnorm Warnings ──
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", message=".*pad_token_id.*")
warnings.filterwarnings("ignore", message=".*HTTP_422.*")
warnings.filterwarnings("ignore", message=".*HF_TOKEN.*")
warnings.filterwarnings("ignore", message=".*clipped samples.*")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

try:
    import transformers
    transformers.logging.set_verbosity_error()
except Exception:
    pass

# Suppress informational flash-attn notice on T4 (T4 natively uses PyTorch SDPA)
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    from qwen_tts import Qwen3TTSModel

# ── PyTorch SDPA & T4 Hardware Optimizations ──
torch.backends.cuda.enable_flash_sdp(False)        # Turing sm_75 does not support FlashAttention-2
torch.backends.cuda.enable_mem_efficient_sdp(True)  # Optimal native SDP kernel on T4
torch.backends.cuda.enable_math_sdp(True)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

MODEL_MAP = {
    "base": "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    "custom": "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",
    "design": "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
}

# ── Smart Multi-Model VRAM Cache ──
_LOADED_MODELS = {}
_PROMPT_CACHE = {}

def get_free_vram_gb():
    if not torch.cuda.is_available():
        return 0.0
    free_bytes, _ = torch.cuda.mem_get_info(0)
    return free_bytes / 1024**3

def clear_gpu_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def load_model(model_type):
    """Loads and caches models in VRAM (float16 + SDPA)."""
    global _LOADED_MODELS
    if model_type in _LOADED_MODELS:
        return _LOADED_MODELS[model_type]

    # If VRAM headroom is tight (< 3.8 GB) and models exist, evict oldest
    if get_free_vram_gb() < 3.8 and _LOADED_MODELS:
        oldest_key = next(iter(_LOADED_MODELS))
        print(f"♻️ VRAM safety: Evicting cached '{oldest_key}' model...")
        del _LOADED_MODELS[oldest_key]
        clear_gpu_cache()

    model_name = MODEL_MAP[model_type]
    print(f"📥 Loading {model_name} (float16 + SDPA)...")
    start = time.time()

    model = Qwen3TTSModel.from_pretrained(
        model_name,
        device_map="cuda:0",
        dtype=torch.float16,
        attn_implementation="sdpa",
    )
    _LOADED_MODELS[model_type] = model
    elapsed = time.time() - start
    allocated = torch.cuda.memory_allocated(0) / 1024**3 if torch.cuda.is_available() else 0
    print(f"✅ Loaded {model_type} (1.7B) in {elapsed:.1f}s | Active VRAM: {allocated:.2f} GB")
    return model

def get_or_create_clone_prompt(model, ref_audio, ref_transcript, use_fast_mode):
    """Reuses extracted voice clone prompt embeddings to accelerate multi-line synthesis."""
    audio_key = ref_audio
    if isinstance(ref_audio, str) and os.path.exists(ref_audio):
        audio_key = f"{ref_audio}_{os.path.getmtime(ref_audio)}"
    cache_key = (audio_key, (ref_transcript or "").strip(), bool(use_fast_mode))

    if cache_key in _PROMPT_CACHE:
        print("⚡ Using cached voice clone prompt embedding")
        return _PROMPT_CACHE[cache_key]

    print("⏱️ Extracting voice clone prompt embedding...")
    prompt_start = time.time()
    if use_fast_mode or not ref_transcript:
        prompt_items = model.create_voice_clone_prompt(
            ref_audio=ref_audio,
            x_vector_only_mode=True
        )
    else:
        prompt_items = model.create_voice_clone_prompt(
            ref_audio=ref_audio,
            ref_text=ref_transcript,
            x_vector_only_mode=False
        )
    print(f"   Prompt created in {time.time() - prompt_start:.2f}s")
    _PROMPT_CACHE[cache_key] = prompt_items
    return prompt_items

# ── Studio Audio Mastering Pipeline (ITU-R BS.1770-4 & Peak Limiting) ──
MASTERING_TARGETS = {
    "YouTube & Streaming (-14 LUFS - Recommended)": -14.0,
    "Podcast & Dialogue (-16 LUFS - Balanced)": -16.0,
    "Maximum Loudness (-12 LUFS - Punchy)": -12.0,
    "Raw Audio (Unmastered)": None,
}

def master_audio(wav, sr, preset_name="YouTube & Streaming (-14 LUFS - Recommended)", max_peak_db=-1.0):
    """
    Studio-grade mastering post-processor:
    1. Removes DC offset
    2. Measures and normalizes integrated loudness to broadcast standard (ITU-R BS.1770-4)
    3. Enforces true-peak limiting at -1.0 dBFS (amplitude 0.891) to guarantee zero distortion
       when imported into video editing software (CapCut, Premiere, DaVinci).
    """
    wav = np.asarray(wav, dtype=np.float32)
    if wav.ndim > 1:
        wav = wav.squeeze()

    # 1. DC Offset Removal
    wav = wav - np.mean(wav)

    target_lufs = MASTERING_TARGETS.get(preset_name, -14.0)
    if target_lufs is None:
        # Raw mode: gentle safety ceiling
        peak = np.max(np.abs(wav))
        if peak > 0.98:
            wav = wav * (0.95 / peak)
        return wav

    # 2. ITU-R BS.1770-4 Loudness Normalization
    try:
        meter = pyln.Meter(sr)
        current_lufs = meter.integrated_loudness(wav)
        if not np.isneginf(current_lufs) and not np.isnan(current_lufs):
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                wav = pyln.normalize.loudness(wav, current_lufs, target_lufs)
        else:
            raise ValueError("Silence or invalid LUFS")
    except Exception:
        # Fallback: Target RMS normalization
        rms = np.sqrt(np.mean(wav**2))
        if rms > 1e-6:
            target_rms = 10 ** (target_lufs / 20.0)
            wav = wav * (target_rms / rms)

    # 3. Peak Limiting & Headroom Protection (-1.0 dBFS = 0.891)
    max_peak = np.max(np.abs(wav))
    target_peak_amp = 10 ** (max_peak_db / 20.0)
    if max_peak > target_peak_amp:
        wav = wav * (target_peak_amp / max_peak)

    return np.clip(wav, -0.99, 0.99)

def save_mastered_audio(wav, sr, preset_name):
    mastered = master_audio(wav, sr, preset_name)
    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
    sf.write(temp_file.name, mastered, sr)
    return temp_file.name

# ── Generation Functions ──
def voice_clone(text, reference_audio, ref_transcript, use_fast_mode, master_preset):
    if not text or not str(text).strip():
        gr.Warning("Please enter text to synthesize.")
        return None, "⚠️ Please enter text to synthesize."
    if not reference_audio:
        gr.Warning("Please provide a reference audio sample.")
        return None, "⚠️ Reference audio is required."

    try:
        model = load_model("base")
        if model is None:
            return None, "❌ Failed to load Base model."

        prompt_items = get_or_create_clone_prompt(model, reference_audio, ref_transcript, use_fast_mode)

        print("⏱️ Generating speech (1.7B)...")
        gen_start = time.time()
        with torch.inference_mode():
            wavs, sr = model.generate_voice_clone(
                text=text.strip(),
                voice_clone_prompt=prompt_items
            )
        gen_time = time.time() - gen_start

        out_path = save_mastered_audio(wavs[0], sr, master_preset)
        audio_dur = len(wavs[0]) / sr
        rtf = gen_time / audio_dur if audio_dur > 0 else 0
        status = f"✅ Synthesized {audio_dur:.1f}s audio in {gen_time:.1f}s (RTF: {rtf:.2f}x) | Model: 1.7B | Mastering: {master_preset.split('(')[0].strip()}"
        print(status)
        return out_path, status
    except Exception as e:
        import traceback
        traceback.print_exc()
        return None, f"❌ Error: {str(e)}"

def custom_voice(text, voice_name, instruction, master_preset):
    if not text or not str(text).strip():
        gr.Warning("Please enter text to synthesize.")
        return None, "⚠️ Please enter text to synthesize."

    try:
        model = load_model("custom")
        if model is None:
            return None, "❌ Failed to load CustomVoice model."

        print(f"⏱️ Generating speech with voice '{voice_name}' (1.7B)...")
        gen_start = time.time()
        with torch.inference_mode():
            if instruction and instruction.strip():
                wavs, sr = model.generate_custom_voice(
                    text=text.strip(),
                    speaker=voice_name,
                    instruct=instruction.strip()
                )
            else:
                wavs, sr = model.generate_custom_voice(
                    text=text.strip(),
                    speaker=voice_name
                )
        gen_time = time.time() - gen_start

        out_path = save_mastered_audio(wavs[0], sr, master_preset)
        audio_dur = len(wavs[0]) / sr
        rtf = gen_time / audio_dur if audio_dur > 0 else 0
        status = f"✅ Synthesized {audio_dur:.1f}s audio in {gen_time:.1f}s (RTF: {rtf:.2f}x) | Voice: {voice_name} (1.7B)"
        print(status)
        return out_path, status
    except Exception as e:
        import traceback
        traceback.print_exc()
        return None, f"❌ Error: {str(e)}"

def voice_design(text, voice_description, master_preset):
    if not text or not str(text).strip():
        gr.Warning("Please enter text to synthesize.")
        return None, "⚠️ Please enter text to synthesize."
    if not voice_description or not str(voice_description).strip():
        gr.Warning("Please enter a voice description.")
        return None, "⚠️ Voice description is required."

    try:
        model = load_model("design")
        if model is None:
            return None, "❌ Failed to load VoiceDesign model."

        print("⏱️ Generating speech with voice design (1.7B)...")
        gen_start = time.time()
        with torch.inference_mode():
            wavs, sr = model.generate_voice_design(
                text=text.strip(),
                instruct=voice_description.strip()
            )
        gen_time = time.time() - gen_start

        out_path = save_mastered_audio(wavs[0], sr, master_preset)
        audio_dur = len(wavs[0]) / sr
        rtf = gen_time / audio_dur if audio_dur > 0 else 0
        status = f"✅ Synthesized {audio_dur:.1f}s audio in {gen_time:.1f}s (RTF: {rtf:.2f}x) | Designed Voice (1.7B)"
        print(status)
        return out_path, status
    except Exception as e:
        import traceback
        traceback.print_exc()
        return None, f"❌ Error: {str(e)}"

def ui_free_vram():
    global _LOADED_MODELS, _PROMPT_CACHE
    _LOADED_MODELS.clear()
    _PROMPT_CACHE.clear()
    clear_gpu_cache()
    free_vram = get_free_vram_gb()
    return f"🧹 VRAM Purged! Free GPU Memory: {free_vram:.2f} GB"

# ── TushKosh 2.0 Standard Gradio Design System ──
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1060px !important; margin: auto !important; }
.brand-header {
    text-align: center;
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    padding: 28px;
    border-radius: 15px;
    margin-bottom: 20px;
    box-shadow: 0 10px 25px rgba(102,126,234,0.3);
}
.brand-title {
    color: white;
    font-size: 2.1em;
    font-weight: 700;
    margin: 0 0 6px 0;
}
.brand-subtitle {
    color: rgba(255,255,255,0.9);
    font-size: 1.05em;
    margin-bottom: 16px;
    text-align: center;
}
.social-buttons {
    display: flex;
    justify-content: center;
    gap: 12px;
    flex-wrap: wrap;
}
.social-btn {
    padding: 9px 22px;
    border-radius: 8px;
    font-weight: 700;
    font-size: 14px;
    text-decoration: none;
    display: inline-block;
    color: white !important;
    transition: all 0.3s;
    box-shadow: 0 4px 12px rgba(0,0,0,0.2);
}
.social-btn:hover {
    transform: translateY(-2px);
    box-shadow: 0 6px 16px rgba(0,0,0,0.3);
}
.youtube-btn { background: linear-gradient(135deg, #FF0000 0%, #CC0000 100%); }
.x-btn { background: linear-gradient(135deg, #000000 0%, #333333 100%); }

#gen-btn, button.primary {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important;
    color: white !important;
    font-weight: 600 !important;
    border-radius: 12px !important;
}
#stop-btn {
    background: linear-gradient(135deg, #ef4444 0%, #b91c1c 100%) !important;
    color: white !important;
    font-weight: 600 !important;
    border-radius: 12px !important;
}
#clear-btn {
    background: linear-gradient(135deg, #6b7280 0%, #374151 100%) !important;
    color: white !important;
    font-weight: 600 !important;
    border-radius: 12px !important;
}
.footer {
    text-align: center;
    padding: 20px;
    margin-top: 30px;
    border-top: 2px solid #e5e7eb;
    color: #6b7280;
}


/* TushKosh 2.0 - Premium Black & Gold Theme */
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700;800&display=swap');
* { font-family: 'Inter', sans-serif !important; }
body, .gradio-container { background: #0A0A0A !important; color: #E8E8E8 !important; }
.gradio-container { max-width: 1050px !important; margin: auto !important; }
h1, h2, h3 { color: #C9A84C !important; }

/* Header & Hero blocks */
.brand-header, .hero-header {
  text-align: center !important;
  background: linear-gradient(135deg, #0A0A0A 0%, #1A1A1A 50%, #0D0D0D 100%) !important;
  border: 1.5px solid #C9A84C !important;
  border-radius: 18px !important;
  padding: 28px 24px !important;
  margin-bottom: 20px !important;
  box-shadow: 0 0 36px rgba(201,168,76,0.18), 0 8px 32px rgba(0,0,0,0.6) !important;
}
.brand-title, .hero-title {
  color: #FFFFFF !important;
  text-align: center !important;
  font-weight: 800 !important;
}
.brand-subtitle, .hero-subtitle {
  color: #C9A84C !important;
  text-align: center !important;
  font-weight: 600 !important;
}
.brand-footer, .footer {
  text-align: center !important;
  color: #888888 !important;
  background: #0D0D0D !important;
  border: 1px solid #C9A84C !important;
  border-radius: 14px !important;
  padding: 16px !important;
  margin-top: 20px !important;
}
.btn-row {
  display: flex !important;
  justify-content: center !important;
  align-items: center !important;
  gap: 10px !important;
  flex-wrap: wrap !important;
}
.social-btn, .yt-btn, .hero-social-btn {
  display: inline-flex !important;
  align-items: center !important;
  justify-content: center !important;
  background: linear-gradient(135deg, #C9A84C 0%, #FFD700 50%, #C9A84C 100%) !important;
  color: #000000 !important;
  font-weight: 800 !important;
  border-radius: 50px !important;
  text-decoration: none !important;
  box-shadow: 0 4px 18px rgba(201,168,76,0.45) !important;
}
.x-btn, .sup-btn { display: none !important; }

button.primary, .gr-button-primary {
  background: linear-gradient(135deg, #C9A84C 0%, #FFD700 50%, #C9A84C 100%) !important;
  color: #000000 !important; font-weight: 800 !important;
  border-radius: 50px !important; border: none !important;
  box-shadow: 0 4px 20px rgba(201,168,76,0.4) !important;
}
input, textarea, select, .gr-input, .gr-textbox {
  background: #111111 !important; border: 1px solid #333333 !important;
  color: #E8E8E8 !important; border-radius: 10px !important;
}
input:focus, textarea:focus {
  border-color: #C9A84C !important;
  box-shadow: 0 0 0 2px rgba(201,168,76,0.2) !important;
}
label, .gr-form > label { color: #C9A84C !important; font-weight: 600 !important; }
.gr-panel, .gr-block, .gr-box, .gr-form {
  background: #111111 !important; border: 1px solid #222222 !important;
  border-radius: 14px !important;
}
input[type=range]::-webkit-slider-thumb { background: #C9A84C !important; }
input[type=range]::-webkit-slider-runnable-track { background: #333 !important; }
"""

with gr.Blocks(
    title="Qwen3-TTS 1.7B - By TushKosh 2.0"
) as demo:

    # Branding Header
    gr.HTML("""
        <div class="brand-header">
            <h1 class="brand-title">🎙️ Qwen3-TTS 1.7B</h1>
            <p class="brand-subtitle">Voice Cloning | Custom Voices | Voice Design | Studio Mastering</p>
            <div class="social-buttons">
                <a href="https://www.youtube.com/@TushKosh2.0?sub_confirmation=1" target="_blank" class="social-btn youtube-btn">
                    ▶ Subscribe to TushKosh 2.0
                </a>
                <a href="https://www.youtube.com/@TushKosh2.0" target="_blank" class="social-btn x-btn">
                    𝕏 Follow on X
                </a>
            </div>
            <p style="margin-top: 12px; font-size: 0.85em; color: rgba(255,255,255,0.75); text-align: center;">
                Notebook by <b>TushKosh 2.0</b> - Studio Quality AI Voice Synthesis
            </p>
        </div>
    """)

    # Studio Audio Mastering Settings
    with gr.Accordion("🎚️ Studio Audio Mastering Settings (Fix Low Volume & Distortion)", open=True):
        master_preset = gr.Dropdown(
            choices=list(MASTERING_TARGETS.keys()),
            value="YouTube & Streaming (-14 LUFS - Recommended)",
            label="Mastering Loudness Preset",
            info="Applies ITU-R BS.1770-4 loudness normalization & -1.0 dBFS true-peak limiter to guarantee zero distortion in video editors."
        )

    DEFAULT_SYNTHESIS_TEXT = "This is the Qwen3-TTS Colab notebook created by TushKosh 2.0 for free tier users. Subscribe to our YouTube channel for more free AI tools and tutorials like this!"

    # ── Tab 1: Voice Cloning ──
    with gr.Tab("🎤 Voice Cloning"):
        gr.Markdown("### 🎤 Zero-Shot Voice Cloning (3+ Seconds Reference Audio)")
        with gr.Row():
            with gr.Column(scale=1):
                clone_text = gr.Textbox(
                    value=DEFAULT_SYNTHESIS_TEXT,
                    label="Text to Synthesize",
                    placeholder="Enter text to speak in the cloned voice...",
                    lines=4
                )
                clone_audio = gr.Audio(label="Reference Audio (3+ seconds)", type="filepath")
                clone_transcript = gr.Textbox(label="Reference Audio Transcript (Optional - improves quality)", placeholder="What is said in the audio...", lines=2)
                clone_fast_mode = gr.Checkbox(label="Fast Mode (Skip transcript parsing)", value=True)
                with gr.Row():
                    clone_btn = gr.Button("🎬 Generate Speech", variant="primary", size="lg", elem_id="gen-btn")
                    clone_stop = gr.Button("🛑 Stop", variant="secondary", size="lg", elem_id="stop-btn")
                    clone_clear = gr.Button("🗑️ Clear", variant="secondary", size="lg", elem_id="clear-btn")
            with gr.Column(scale=1):
                clone_output = gr.Audio(label="Mastered Speech Output", type="filepath")
                clone_status = gr.Textbox(label="Status & Diagnostics", interactive=False)
                gr.Markdown("💡 **Speed Tip**: First run caches weights in VRAM. Subsequent runs with the same reference sample reuse embeddings instantly!")

        clone_event = clone_btn.click(
            fn=voice_clone,
            inputs=[clone_text, clone_audio, clone_transcript, clone_fast_mode, master_preset],
            outputs=[clone_output, clone_status]
        )
        clone_stop.click(fn=None, cancels=[clone_event])
        clone_clear.click(
            fn=lambda: ("", None, "", True, None, ""),
            outputs=[clone_text, clone_audio, clone_transcript, clone_fast_mode, clone_output, clone_status]
        )

    # ── Tab 2: Custom Voice ──
    with gr.Tab("🎭 Custom Voice"):
        gr.Markdown("### 🎭 9 High-Fidelity Character Voices with Style Instructions")
        with gr.Row():
            with gr.Column(scale=1):
                custom_text = gr.Textbox(
                    value=DEFAULT_SYNTHESIS_TEXT,
                    label="Text to Synthesize",
                    placeholder="Enter text to synthesize...",
                    lines=3
                )
                custom_voice_name = gr.Dropdown(
                    choices=["Serena", "Vivian", "Ono_Anna", "Sohee", "Aiden", "Dylan", "Eric", "Ryan", "Uncle_Fu"],
                    label="Voice Character",
                    value="Serena"
                )
                custom_instruction = gr.Textbox(label="Style & Emotion Instruction (Optional)", placeholder="e.g. 'Speak warmly and enthusiastically', 'Whisper softly'", lines=2)
                gr.Markdown("**Female**: Serena, Vivian, Ono_Anna, Sohee · **Male**: Aiden, Dylan, Eric, Ryan, Uncle_Fu")
                with gr.Row():
                    custom_btn = gr.Button("🎬 Generate Speech", variant="primary", size="lg", elem_id="gen-btn")
                    custom_stop = gr.Button("🛑 Stop", variant="secondary", size="lg", elem_id="stop-btn")
                    custom_clear = gr.Button("🗑️ Clear", variant="secondary", size="lg", elem_id="clear-btn")
            with gr.Column(scale=1):
                custom_output = gr.Audio(label="Mastered Speech Output", type="filepath")
                custom_status = gr.Textbox(label="Status & Diagnostics", interactive=False)

        custom_event = custom_btn.click(
            fn=custom_voice,
            inputs=[custom_text, custom_voice_name, custom_instruction, master_preset],
            outputs=[custom_output, custom_status]
        )
        custom_stop.click(fn=None, cancels=[custom_event])
        custom_clear.click(
            fn=lambda: ("", "Serena", "", None, ""),
            outputs=[custom_text, custom_voice_name, custom_instruction, custom_output, custom_status]
        )

    # ── Tab 3: Voice Design ──
    with gr.Tab("🎨 Voice Design"):
        gr.Markdown("### 🎨 Design Unique Voices with Natural Language Descriptions (1.7B)")
        with gr.Row():
            with gr.Column(scale=1):
                design_text = gr.Textbox(
                    value=DEFAULT_SYNTHESIS_TEXT,
                    label="Text to Synthesize",
                    placeholder="Enter text to synthesize...",
                    lines=3
                )
                design_description = gr.Textbox(
                    value="A young cheerful female speaking clearly with a gentle British accent and a warm, friendly smile.",
                    label="Voice Description",
                    placeholder="e.g. 'A young cheerful female speaking clearly with a gentle British accent', 'A seasoned deep male narrator'",
                    lines=3
                )
                gr.Markdown("💡 **Prompting Tip**: Describe age, gender, accent, tone, pitch, pacing, and emotional attitude.")
                with gr.Row():
                    design_btn = gr.Button("🎬 Generate Speech", variant="primary", size="lg", elem_id="gen-btn")
                    design_stop = gr.Button("🛑 Stop", variant="secondary", size="lg", elem_id="stop-btn")
                    design_clear = gr.Button("🗑️ Clear", variant="secondary", size="lg", elem_id="clear-btn")
            with gr.Column(scale=1):
                design_output = gr.Audio(label="Mastered Speech Output", type="filepath")
                design_status = gr.Textbox(label="Status & Diagnostics", interactive=False)

        design_event = design_btn.click(
            fn=voice_design,
            inputs=[design_text, design_description, master_preset],
            outputs=[design_output, design_status]
        )
        design_stop.click(fn=None, cancels=[design_event])
        design_clear.click(
            fn=lambda: ("", "", None, ""),
            outputs=[design_text, design_description, design_output, design_status]
        )

    # ── VRAM Management Utilities ──
    with gr.Row():
        vram_btn = gr.Button("🧹 Free VRAM Cache", variant="secondary", size="sm")
        vram_status = gr.Textbox(label="Memory Status", interactive=False, lines=1)
    vram_btn.click(fn=ui_free_vram, outputs=vram_status)

    # Branding Footer
    gr.HTML("""
        <div class="footer">
            <p style="font-size: 16px; margin: 5px 0;">🎙️ Created by <strong>TushKosh 2.0</strong></p>
            <p style="font-size: 14px; margin: 5px 0; color: #9ca3af;">Free &amp; Open Source | Qwen3-TTS 1.7B | Colab T4 GPU Edition</p>
            <p style="font-size: 13px; margin: 10px 0;">
                <a href="https://www.youtube.com/@TushKosh2.0?sub_confirmation=1" target="_blank" style="color: #667eea; text-decoration: none; margin: 0 10px; font-weight: 600;">YouTube</a> | 
                <a href="https://www.youtube.com/@TushKosh2.0" target="_blank" style="color: #667eea; text-decoration: none; margin: 0 10px; font-weight: 600;">X (Twitter)</a>
            </p>
        </div>
    """)

print("=" * 60)
print("🎙️ Qwen3-TTS 1.7B - Web UI Ready")
print("📺 Created by: TushKosh 2.0")
print("🔗 YouTube: @TushKosh2.0 | X: @TushKosh2.0")
print("=" * 60)
print("\n🚀 Launching Gradio Web UI...")
demo.queue()
demo.launch(share=True, inline=False, debug=True, show_error=True, theme=gr.themes.Soft(), css=CSS)

<hr style="border:none;border-top:1px solid #C9A84C;margin:30px 0 20px;" /><div align="center" style="background:linear-gradient(135deg,#0A0A0A 0%,#111111 60%,#0D0D0D 100%);border:1px solid #C9A84C;border-radius:16px;padding:28px 24px 24px;margin:0 0 10px;box-shadow:0 0 30px rgba(201,168,76,0.12),0 4px 24px rgba(0,0,0,0.5);font-family:Arial,sans-serif;text-align:center;"><div style="width:60px;height:2px;background:linear-gradient(90deg,transparent,#C9A84C,#FFD700,#C9A84C,transparent);margin:0 auto 16px;border-radius:2px;"></div><h2 align="center" style="color:#FFFFFF;margin:0 0 6px;font-size:1.5em;letter-spacing:2px;text-align:center;">TushKosh 2.0</h2><p align="center" style="color:#C9A84C;margin:0 0 20px;font-weight:700;letter-spacing:3px;font-size:12px;text-transform:uppercase;text-align:center;">Learn &bull; Build &bull; Create</p><div align="center" style="text-align:center;margin:0 0 18px;"><a href="https://www.youtube.com/@TushKosh2.0?sub_confirmation=1" target="_blank" style="display:inline-block;background:linear-gradient(135deg,#C9A84C 0%,#FFD700 50%,#C9A84C 100%);color:#000000;font-weight:800;font-size:13px;padding:11px 28px;border-radius:50px;text-decoration:none;letter-spacing:1px;box-shadow:0 4px 18px rgba(201,168,76,0.45);">&#9654; Subscribe on YouTube</a></div><p align="center" style="color:#888888;font-size:12px;margin:0 0 6px;text-align:center;">Original notebook by <strong style="color:#AAAAAA;">AIQUEST Academy</strong></p><p align="center" style="color:#666666;font-size:11px;margin:0;text-align:center;">TushKosh 2.0 branding added &middot; Original code and third-party components attributed to their respective creators.</p><div style="width:60px;height:2px;background:linear-gradient(90deg,transparent,#C9A84C,#FFD700,#C9A84C,transparent);margin:16px auto 0;border-radius:2px;"></div></div>